# =============================================================
# SIMULADOR FROZEN — 05b (MODELO FINAL EM PRODUÇÃO)
# =============================================================
### Previsão genuína e independente do torneio em curso.
### Features de forma congeladas a 2026-06-10 (df_wc_predict.csv) - Zero data leakage — este é o simulador correcto para previsão pura do Mundial 2026.
#
### Usa modelos V4 (Diff_Points + sample_weight).
# =============================================================

In [4]:
import pandas as pd
import numpy as np
import joblib

In [5]:
# --- 1. Carregar dados e modelos ---
df_wc_predict = pd.read_csv("../data/df_wc_predict.csv")
df_wc_predict["date"] = pd.to_datetime(df_wc_predict["date"])
df_ranking = pd.read_csv("../data/fifa_ranking.csv")

clf_lr = joblib.load("../models/model_v4_clf_lr.pkl")
clf_rf = joblib.load("../models/model_v4_clf_rf.pkl")

class EnsembleClassifier:
    def __init__(self, clf_predict, clf_proba):
        self.clf_predict = clf_predict
        self.clf_proba   = clf_proba
        self.classes_    = clf_proba.classes_

    def predict(self, X):
        return self.clf_predict.predict(X)

    def predict_proba(self, X):
        return self.clf_proba.predict_proba(X)

clf = EnsembleClassifier(clf_lr, clf_rf)

FEATURES = [
    "Diff_Points", "Fator_Casa",
    "Forma_Golos_Home", "Forma_Pts_Home", "Forma_Golos_Sofridos_Home",
    "Forma_Golos_Away", "Forma_Pts_Away", "Forma_Golos_Sofridos_Away",
    "Tipo_Competicao",
]

# --- 2. Grupos ---
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Ecuador", "Ivory Coast", "Curaçao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Colombia", "DR Congo", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

TEAM_GROUP = {team: grp for grp, teams in GROUPS.items() for team in teams}

# --- 3. Ranking FIFA para desempate ---
NAME_MAP = {
    "south korea": "korea republic", "north korea": "dpr korea",
    "iran": "ir iran", "china": "china pr", "dr congo": "congo dr",
    "czech republic": "czechia", "gambia": "the gambia",
    "kyrgyzstan": "kyrgyz republic", "cape verde": "cabo verde",
    "ivory coast": "côte d'ivoire", "turkey": "türkiye",
    "united states": "usa", "taiwan": "chinese taipei",
    "saint kitts and nevis": "st. kitts and nevis",
    "saint lucia": "st. lucia",
    "saint vincent and the grenadines": "st. vincent / grenadines",
    "united states virgin islands": "us virgin islands",
}
df_ranking["name_norm"] = df_ranking["country_name"].str.strip().str.lower()
rank_lookup = df_ranking.set_index("name_norm")["ranking"]
rank_lookup_full = df_ranking[["name_norm", "ranking", "points"]].set_index("name_norm")

def get_rank(team):
    key = NAME_MAP.get(team.lower(), team.lower())
    return rank_lookup.get(key, 999)

def get_points(team):
    key = NAME_MAP.get(team.lower(), team.lower())
    return rank_lookup_full["points"].get(key, np.nan)

# --- 4. Inicializar tabela de pontos ---
def init_table(groups):
    records = []
    for grp, teams in groups.items():
        for team in teams:
            records.append({
                "group": grp, "team": team,
                "pts": 0, "gf": 0, "ga": 0, "gd": 0,
                "ranking": get_rank(team)
            })
    return pd.DataFrame(records)

table = init_table(GROUPS)

def update_table(table, home, away, result):
    if result == "H":
        table.loc[table["team"] == home, "pts"] += 3
    elif result == "D":
        table.loc[table["team"] == home, "pts"] += 1
        table.loc[table["team"] == away, "pts"] += 1
    else:
        table.loc[table["team"] == away, "pts"] += 3
    return table

# --- 5. Simular fase de grupos ---
played = pd.DataFrame()
future = df_wc_predict.copy()
future["Fator_Casa"]      = 0
future["Tipo_Competicao"] = "Mundial"

print("MODO: Simulação totalmente preditiva (features congeladas a 2026-06-10)")
print(f"Jogos a prever: {len(future)}")

predicted_results = []
for _, row in future.iterrows():
    X = pd.DataFrame([row[FEATURES]])
    result = clf.predict(X)[0]
    predicted_results.append({
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "predicted": result
    })
    table = update_table(table, row["home_team"], row["away_team"], result)

df_predicted = pd.DataFrame(predicted_results)

# --- 6. Classificação final dos grupos ---
table_sorted = (
    table
    .sort_values(["group", "pts", "ranking"], ascending=[True, False, True])
    .reset_index(drop=True)
)
table_sorted["position"] = table_sorted.groupby("group").cumcount() + 1

print("\n=== CLASSIFICAÇÃO FINAL DOS GRUPOS ===")
for grp in sorted(GROUPS.keys()):
    grp_df = table_sorted[table_sorted["group"] == grp][["position","team","pts","ranking"]]
    print(f"\nGrupo {grp}:")
    print(grp_df.to_string(index=False))

MODO: Simulação totalmente preditiva (features congeladas a 2026-06-10)
Jogos a prever: 72

=== CLASSIFICAÇÃO FINAL DOS GRUPOS ===

Grupo A:
 position           team  pts  ranking
        1         Mexico    9       11
        2    South Korea    6       24
        3 Czech Republic    3       44
        4   South Africa    0       61

Grupo B:
 position                   team  pts  ranking
        1            Switzerland    9       17
        2                 Canada    6       29
        3 Bosnia and Herzegovina    3       64
        4                  Qatar    0       57

Grupo C:
 position     team  pts  ranking
        1  Morocco    9        6
        2   Brazil    6        5
        3 Scotland    3       41
        4    Haiti    0       87

Grupo D:
 position          team  pts  ranking
        1 United States    9       13
        2     Australia    6       26
        3        Turkey    3       32
        4      Paraguay    0       37

Grupo E:
 position        team  pts  rankin

Quase todos os grupos têm o 1º com 9 pontos e o 2º com 6. Isto significa que o modelo raramente prevê empates e favorece sistematicamente o melhor ranked — o que é consistente com a feature importance que vimos (Diff_Ranking a 27%).
Em 12 grupos, apenas o Grupo C tem uma "surpresa" real (Marrocos acima do Brasil). Os restantes 11 seguem o ranking à risca.
Isto é simultaneamente uma força e uma limitação do modelo — é consistente mas pouco surpreendente. O futebol real tem muito mais variância.

In [6]:
# --- 7. Extrair qualificados e terceiros ---
qualified = {}
thirds = []

for grp in sorted(GROUPS.keys()):
    grp_df = table_sorted[table_sorted["group"] == grp].sort_values(
        ["pts", "ranking"], ascending=[False, True]
    )
    qualified[f"1{grp}"] = grp_df.iloc[0]["team"]
    qualified[f"2{grp}"] = grp_df.iloc[1]["team"]
    thirds.append({
        "group":   grp,
        "team":    grp_df.iloc[2]["team"],
        "pts":     grp_df.iloc[2]["pts"],
        "ranking": grp_df.iloc[2]["ranking"],
    })

df_thirds = (
    pd.DataFrame(thirds)
    .sort_values(["pts", "ranking"], ascending=[False, True])
    .reset_index(drop=True)
)
print("\n=== TERCEIROS CLASSIFICADOS (ordenados) ===")
print(df_thirds.to_string(index=False))

best8_thirds = df_thirds.head(8)
worst4_thirds = df_thirds.iloc[8:]
print(f"\nEliminados (4 piores terceiros):")
print(worst4_thirds[["group","team","pts"]].to_string(index=False))

used_thirds = []

def assign_third(pool, used_thirds, best8):
    candidates = best8[best8["group"].isin(pool) & ~best8["team"].isin(used_thirds)]
    if candidates.empty:
        candidates = best8[~best8["team"].isin(used_thirds)]
    return candidates.iloc[0]["team"]

def get_third(pool):
    team = assign_third(pool, used_thirds, best8_thirds)
    used_thirds.append(team)
    return team

# --- 8. Bracket fase de 32 ---
bracket_32 = {
    73: (qualified["2A"], qualified["2B"]),
    74: (qualified["1E"], get_third(["A","B","C","D","F"])),
    75: (qualified["1F"], qualified["2C"]),
    76: (qualified["1C"], qualified["2F"]),
    77: (qualified["1I"], get_third(["C","D","F","G","H"])),
    78: (qualified["2E"], qualified["2I"]),
    79: (qualified["1A"], get_third(["C","E","F","H","I"])),
    80: (qualified["1L"], get_third(["E","H","I","J","K"])),
    81: (qualified["1D"], get_third(["B","E","F","I","J"])),
    82: (qualified["1G"], get_third(["A","E","H","I","J"])),
    83: (qualified["2K"], qualified["2L"]),
    84: (qualified["1H"], qualified["2J"]),
    85: (qualified["1B"], get_third(["E","F","G","I","J"])),
    86: (qualified["1J"], qualified["2H"]),
    87: (qualified["1K"], get_third(["D","E","I","J","L"])),
    88: (qualified["2D"], qualified["2G"]),
}

print("\n=== FASE DE 32 ===")
for jogo, (home, away) in bracket_32.items():
    print(f"  Jogo {jogo}: {home} vs {away}")

# --- 9. Função simulate_ko ---
def simulate_ko(home, away):
    X = pd.DataFrame([{
        "Diff_Points":               get_points(home) - get_points(away),
        "Fator_Casa":                0,
        "Forma_Golos_Home":          df_wc_predict[df_wc_predict["home_team"] == home]["Forma_Golos_Home"].iloc[0] if len(df_wc_predict[df_wc_predict["home_team"] == home]) > 0 else 1.4,
        "Forma_Pts_Home":            df_wc_predict[df_wc_predict["home_team"] == home]["Forma_Pts_Home"].iloc[0] if len(df_wc_predict[df_wc_predict["home_team"] == home]) > 0 else 1.4,
        "Forma_Golos_Sofridos_Home": df_wc_predict[df_wc_predict["home_team"] == home]["Forma_Golos_Sofridos_Home"].iloc[0] if len(df_wc_predict[df_wc_predict["home_team"] == home]) > 0 else 1.4,
        "Forma_Golos_Away":          df_wc_predict[df_wc_predict["away_team"] == away]["Forma_Golos_Away"].iloc[0] if len(df_wc_predict[df_wc_predict["away_team"] == away]) > 0 else 1.4,
        "Forma_Pts_Away":            df_wc_predict[df_wc_predict["away_team"] == away]["Forma_Pts_Away"].iloc[0] if len(df_wc_predict[df_wc_predict["away_team"] == away]) > 0 else 1.4,
        "Forma_Golos_Sofridos_Away": df_wc_predict[df_wc_predict["away_team"] == away]["Forma_Golos_Sofridos_Away"].iloc[0] if len(df_wc_predict[df_wc_predict["away_team"] == away]) > 0 else 1.4,
        "Tipo_Competicao":           "Mundial",
    }])

    classes = list(clf.classes_)
    proba   = clf.predict_proba(X)[0]
    p_home  = proba[classes.index("H")]
    p_away  = proba[classes.index("A")]
    total   = p_home + p_away
    p_home_norm = p_home / total
    winner  = home if p_home_norm >= 0.5 else away
    return winner, round(p_home_norm, 3), round(1 - p_home_norm, 3)

# --- 10. Simular todas as fases ---
def simulate_round(matchups, round_name):
    print(f"\n=== {round_name} ===")
    winners = {}
    losers  = {}
    for jogo, (home, away) in matchups.items():
        winner, ph, pa = simulate_ko(home, away)
        loser = away if winner == home else home
        winners[jogo] = winner
        losers[jogo]  = loser
        print(f"  Jogo {jogo}: {home} vs {away} → {winner} ({ph} vs {pa})")
    return winners, losers

w32, l32 = simulate_round(bracket_32, "FASE DE 32")

bracket_16 = {
    89: (w32[74], w32[77]),
    90: (w32[73], w32[75]),
    91: (w32[76], w32[78]),
    92: (w32[79], w32[80]),
    93: (w32[83], w32[84]),
    94: (w32[81], w32[82]),
    95: (w32[86], w32[88]),
    96: (w32[85], w32[87]),
}
w16, _ = simulate_round(bracket_16, "OITAVOS DE FINAL")

bracket_qf = {
    97:  (w16[89], w16[90]),
    98:  (w16[93], w16[94]),
    99:  (w16[91], w16[92]),
    100: (w16[95], w16[96]),
}
wqf, _ = simulate_round(bracket_qf, "QUARTOS DE FINAL")

bracket_sf = {
    101: (wqf[97],  wqf[98]),
    102: (wqf[99],  wqf[100]),
}
wsf, lsf = simulate_round(bracket_sf, "MEIAS-FINAIS")

bracket_3rd = {103: (lsf[101], lsf[102])}
w3rd, _ = simulate_round(bracket_3rd, "DISPUTA DO 3º LUGAR")

bracket_final = {104: (wsf[101], wsf[102])}
wfinal, _ = simulate_round(bracket_final, "FINAL")

print(f"\n{'='*50}")
print(f"🏆 CAMPEÃO DO MUNDO 2026: {wfinal[104]}")
print(f"🥈 Vice-campeão:          {[t for t in [wsf[101],wsf[102]] if t != wfinal[104]][0]}")
print(f"🥉 3º lugar:              {w3rd[103]}")


=== TERCEIROS CLASSIFICADOS (ordenados) ===
group                   team  pts  ranking
    I                Senegal    3       19
    J                Austria    3       23
    G                  Egypt    3       27
    E                Ecuador    3       30
    D                 Turkey    3       32
    L                 Panama    3       40
    C               Scotland    3       41
    K               DR Congo    3       43
    A         Czech Republic    3       44
    F                Tunisia    3       58
    H             Cape Verde    3       63
    B Bosnia and Herzegovina    3       64

Eliminados (4 piores terceiros):
group                   team  pts
    A         Czech Republic    3
    F                Tunisia    3
    H             Cape Verde    3
    B Bosnia and Herzegovina    3

=== FASE DE 32 ===
  Jogo 73: South Korea vs Canada
  Jogo 74: Germany vs Turkey
  Jogo 75: Netherlands vs Brazil
  Jogo 76: Morocco vs Japan
  Jogo 77: France vs Egypt
  Jogo 78: Ivory Coast

Argentina campeã em ambos (V3 e V4) — resultado estável

Final idêntica: Argentina vs Espanha

Diferença na convicção: V4 dá Argentina 0.635 vs V3 que dava 0.561 — o V4 é mais confiante
3º lugar diferente: Inglaterra (V4) vs era Bélgica (V3 anterior)

Grupo I — surpresa interessante:

Noruega (#22) em 2º acima do Senegal (#19) — ranking quase igual mas forma pré-Mundial de Noruega superior
Senegal só fica em 3º com 3 pontos

Grupo J:

Argélia (#28) em 2º acima da Áustria (#23) — ranking favorecia Áustria mas forma decide

Conclusão do V4:

O resultado é consistente com o V3 frozen — Argentina e Espanha na final. O Diff_Points e o sample_weight não alteraram o campeão mas produziram probabilidades diferentes e alguns grupos com resultados distintos.